# Laboratorio 4 — Parte 2: modelos de Machine Learning sobre datos geoespaciales
## Lagos Atitlán y Amatitlán — Ejercicios 1 a 10 (entrega completa)

Este notebook continúa el trabajo de `Lab4.ipynb` (Parte I). A partir de los mismos rasters Sentinel-2 L2A (mismos lagos, mismas 11 fechas oficiales por lago), se construye un **conjunto de datos tabular** apto para Machine Learning, se define la **variable respuesta binaria** de alta presencia de cianobacteria, y se seleccionan/justifican las **variables predictoras**.

**Alcance:** los 10 ejercicios de la guía `Laboratorio_4_Parte_2_Datos_Geoespaciales_2026.md` — preparación de datos, variable respuesta, selección de predictores, modelos (Regresión Logística, Random Forest, Gradient Boosting/XGBoost), evaluación, validación espacial y temporal, generalización entre lagos, interpretabilidad (SHAP) y mapas predictivos.


## 0. Dependencias y configuración

Se reutilizan las mismas dependencias de la Parte I. Si el caché de escenas (`data/GIS/lab4_avance/*.tif`) generado en la Parte I ya existe en este equipo, se reutiliza directamente; en caso contrario, se descarga de nuevo desde openEO (requiere autenticación OIDC con cuenta Copernicus).

In [96]:
from datetime import date, timedelta
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import openeo
import pandas as pd
import rasterio
from rasterio.warp import transform as warp_transform

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 30)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data" / "GIS" / "lab4_avance"       # cache de escenas de la Parte I (se reutiliza)
OUT_DIR = BASE_DIR / "data" / "GIS" / "lab4_parte2"          # salidas propias de esta Parte II
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://openeo.dataspace.copernicus.eu"
COLLECTION = "SENTINEL2_L2A"
BANDS = ["B02", "B03", "B04", "B05", "B08", "SCL"]

# Paso del muestreo sistemático (en píxeles nativos, ~10 m/píxel). Ver justificación en el Ejercicio 1.6.
SAMPLE_STRIDE_PX = 20

print(f"Cache de escenas (Parte I): {DATA_DIR.resolve()}")
print(f"Salidas de esta Parte II: {OUT_DIR.resolve()}")

Cache de escenas (Parte I): /Users/dijan/Documents/U/Data Science/Lab4/data/GIS/lab4_avance
Salidas de esta Parte II: /Users/dijan/Documents/U/Data Science/Lab4/data/GIS/lab4_parte2


## 1. Áreas y fechas oficiales (idénticas a la Parte I)

La guía de la Parte II exige usar las mismas fechas que en la Parte I. Se copian exactamente los mismos diccionarios `LAKES` y `OFFICIAL_DATES` de `Lab4.ipynb`.

In [97]:
LAKES = {
    "Atitlan": {
        "west": -91.326256, "east": -91.07151,
        "south": 14.5948, "north": 14.750979,
    },
    "Amatitlan": {
        "west": -90.638065, "east": -90.512924,
        "south": 14.412347, "north": 14.493799,
    },
}

OFFICIAL_DATES = {
    "Atitlan": [
        "2025-01-18", "2025-04-13", "2025-05-13",
        "2025-07-17", "2025-11-21", "2025-12-29",
        "2026-02-12", "2026-03-24", "2026-04-13",
        "2026-04-28", "2026-07-22",
    ],
    "Amatitlan": [
        "2025-01-28", "2025-04-15", "2025-04-28",
        "2025-11-24", "2026-01-08", "2026-02-02",
        "2026-02-07", "2026-03-29", "2026-04-13",
        "2026-04-28", "2026-06-19",
    ],
}

assert set(LAKES) == set(OFFICIAL_DATES)
assert all(len(v) == 11 for v in OFFICIAL_DATES.values())
print("Configuración validada: 2 lagos y 11 fechas oficiales por lago (idénticas a la Parte I).")

Configuración validada: 2 lagos y 11 fechas oficiales por lago (idénticas a la Parte I).
